Assignment for day 4

1. MC Paths under GBM model
2. Kelly Criterion
3. Maximize Utility Function

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import minimize

# make all plotly charts have the same size
px.defaults.width = 800
px.defaults.height = 500


### MC PATHS

In [2]:
# function to generate GBM paths
def mc_gbm(S0, mu, sigma, T, dt, N):
    """
    Geometric Brownian Motion Monte Carlo
    S0: initial price | mu: drift | sigma: vol
    T: time horizon   | dt: timestep | N: paths
    """
    n_steps = int(T / dt)
    # Random shocks: shape (N_paths, N_steps)
    Z = np.random.standard_normal((N, n_steps))
    
    # Log returns per step
    log_ret = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    
    # Cumulative product → price paths
    paths = S0 * np.exp(np.cumsum(log_ret, axis=1))
    return paths

# Example: 10,000 paths, 1 year, daily steps
paths = mc_gbm(S0=100, mu=0.08, sigma=0.20,
               T=1.0, dt=1/252, N=10_000)

print(f"Expected final price: {paths[:,-1].mean():.2f}")
print(f"5th percentile:       {np.percentile(paths[:,-1], 5):.2f}")
print(f"95th percentile:      {np.percentile(paths[:,-1], 95):.2f}")


Expected final price: 108.03
5th percentile:       76.00
95th percentile:      146.87


In [3]:
df_paths = pd.DataFrame(paths).T

In [4]:
# plot paths
fig = px.line(df_paths.iloc[:, :25], title="GBM Paths")
# rename axes
fig.update_yaxes(title_text="Price")
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text="Time")
fig.show()

# plot histogram of final prices
fig = px.histogram(paths[:,-1], title="Histogram of Final Prices")
fig.update_xaxes(title_text="Final Price")
fig.update_layout(showlegend=False)
fig.show()

In [5]:
# calculate and plot dradowns for first 5 paths
df_drawdowns = df_paths - df_paths.cummax()

fig = px.line(df_drawdowns.iloc[:, :5], title="Drawdowns for First 5 Paths")
fig.update_xaxes(title_text="Time")
fig.update_yaxes(title_text="Negative Drawdown")
fig.update_layout(showlegend=False)
fig.show()

# calculate and plot %of time in drawdown for first 5 paths
df_drawdown_duration = df_drawdowns.apply(lambda x: x.eq(0).astype(int).cumsum())/len(df_drawdowns)
# plot drawdown duration
fig = px.line(df_drawdown_duration.iloc[:, :5], title="Drawdown Duration for First 5 Paths")
fig.update_layout(showlegend=False)
fig.update_xaxes(title_text="Time")
fig.update_yaxes(title_text="% of time in drawdown")
fig.show()


### KELLY CRITERION

In [6]:
# write a function to calculate the kelly fraction
def kelly_fraction(b, p):
    q = 1 - p
    return max(0, (b*p - q) / b)    # floor at 0 (ignore negative expected returns)

# calculate kelly fraction for different values of b and p as a dataframe, with b as rows and p as columns
bs = np.linspace(0.5, 2.0, 100)
ps = np.linspace(0, 1, 100)
fracs = []
for b in bs:
    for p in ps:
        fracs.append(kelly_fraction(b, p))
fracs = np.array(fracs).reshape(len(bs), len(ps))
df_fracs = pd.DataFrame(fracs, index=bs, columns=ps)

# plot kelly fraction as heatmap
fig = px.imshow(df_fracs, title="Kelly Fraction")
fig.update_xaxes(title_text="p")
fig.update_yaxes(title_text="b")
fig.show()


### UTILITY FUNCTION

In [7]:
# utility function with linear, quadratic and logarithmic terms
# U(x) = ax - bx^2 - cx^2 - d ln(x)
def utility_function(x, a, b, c, d):
    return a*x - b*x**2 - c*x**2 - d*np.log(x)

def negative_utility_function(x, a, b, c, d):
    return -utility_function(x, a, b, c, d)

# call scipy.optimize.minimize to find the maximum of the utility function (= minium of the negative utility function)
res = minimize(negative_utility_function, x0=0.25, args=(1, 1, 0.01, 0.05))

res.x

array([0.43861745])